# Bulk observables: FastHydro's realistic medium vs MUSIC + 3D MC-Glauber

`config/fasthydro_wake_realistic.yaml` normalizes fast_data's tilted MC-Glauber so that **one**
number comes out right: $dN_{ch}/d\eta \approx 650$ at $b = 0$ (`initial_state.target_T = 0.39`).
Nothing else about that initial state is tuned: not its transverse granularity, its
longitudinal profile, or how it scales with $b$. This notebook asks how realistic it is. It
compares against the calibrated 3D MC-Glauber + MUSIC setup of
`config/AuAu_MCGlauber_MUSIC_0_10.xml`, in the bulk observables: $p_T$ spectra,
$\langle p_T\rangle$ and $dN/d\eta$.

On a **shared** initial condition FastHydro and MUSIC agree to 1–3 % (`config/FVvsMUSIC`). What
separates them here is therefore mainly the initial state.

| model | chain | file |
|---|---|---|
| **MUSIC calibrated** | 3D MC-Glauber strings → MUSIC ($\eta/s(T)$, $\zeta/s(T)$) → iSS with $\delta f$ | `hadrons_music_calibrated.npz` |
| **MUSIC matched** *(optional)* | the same strings → MUSIC with FastHydro's transport ($\eta/s = 0.08$, no bulk) → iSS without $\delta f$ | `hadrons_music_matched.npz` |
| **FastHydro** | fast_data MC-Glauber (`target_T` 0.39) → FastHydro (Israel–Stewart) → iSS | `hadrons_fasthydro.npz` |
| FastHydro, $b = 0$ *(if present)* | the tuning point itself: the hadron-wake notebook's background run | `out_hadron_wake/hadrons_bg.npz` |

Every model particlizes at $T = 0.15$ GeV on the same EoS: hotQCD, MUSIC's EOS 9, UrQMD hadron
list. iSS decays the resonances; there is no hadronic afterburner. **0–10 % is $b \in [0, 4.7]$ fm
in both Glauber models** (§1).

What each pair measures:
- **MUSIC matched vs FastHydro** differ only in the initial state, including the 3D Glauber's
  dynamical string deposition. This is the initial state's effect on its own.
- **MUSIC calibrated vs matched** is what the calibrated transport and $\delta f$ add.
- **MUSIC calibrated vs data** is how far the reference itself is from experiment, without SMASH.

Make the files with **`example/make_bulk_comparison_data.py`** (§1 prints the command).

In [ ]:
import json, os, sys
import numpy as np
import matplotlib.pyplot as plt

for p in (os.path.join("..", "python"), os.path.join(os.getcwd(), "..", "python")):
    if os.path.isdir(p) and p not in sys.path:
        sys.path.insert(0, p)
from fasthydro.hadrons import load

OUT = os.environ.get("BULK_VS_MUSIC_OUT", "out_bulk_vs_music")
WAKE_OUT = os.environ.get("HADRON_WAKE_OUT",
                          os.path.join(os.path.dirname(os.path.abspath(OUT)), "out_hadron_wake"))

# --- one fixed visual language ---------------------------------------------------------------
# Hues are ASSIGNED to models, never cycled (Okabe-Ito), and every model also has its own line
# style and marker, so identity never rests on colour alone.
MODELS = {
    "music_calibrated": dict(label="MUSIC + 3D Glauber, calibrated", color="#222222", ls="-",  marker="o"),
    "music_matched":    dict(label="MUSIC + 3D Glauber, FastHydro transport", color="#0072B2", ls="--", marker="s"),
    "fasthydro":        dict(label="FastHydro, realistic IC, 0-10%", color="#D55E00", ls="-.", marker="^"),
    "fasthydro_b0":     dict(label="FastHydro, realistic IC, b = 0", color="#CC79A7", ls=":",  marker="v"),
}
FILES = {"music_calibrated": (OUT, "music_calibrated"), "music_matched": (OUT, "music_matched"),
         "fasthydro": (OUT, "fasthydro"), "fasthydro_b0": (WAKE_OUT, "bg")}
REF = "music_calibrated"       # every ratio is to this model

def tidy(ax, xlabel=None, ylabel=None, title=None):
    ax.grid(True, alpha=0.25, lw=0.6); ax.set_axisbelow(True)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    if xlabel: ax.set_xlabel(xlabel)
    if ylabel: ax.set_ylabel(ylabel)
    if title: ax.set_title(title, loc="left", fontsize=11)
    return ax

def band(ax, x, y, e, **kw):
    # a line with its 1-sigma band: the band in the line's own colour, quieter
    ln, = ax.plot(x, y, **kw)
    if e is not None and np.isfinite(e).any():
        ax.fill_between(x, y - e, y + e, color=ln.get_color(), alpha=0.18, lw=0)
    return ln

def centers(e):
    return 0.5 * (e[1:] + e[:-1])

def style(m, marker=False):
    s = MODELS[m]
    kw = dict(color=s["color"], ls=s["ls"], lw=1.7, label=s["label"])
    if marker:
        kw.update(marker=s["marker"], ms=4)
    return kw

plt.rcParams.update({"figure.dpi": 110, "font.size": 10, "axes.titlesize": 11})

### Errors: the event average, not the oversampling

The hadron-wake notebook's errors are iSS's compound-Poisson noise. That is right there,
because its jet and background runs share each event's initial condition. Here each model
averages its own set of fluctuating 0–10 % events, and the event-to-event spread dominates:
$b$ alone moves $dN_{ch}/d\eta$ by ~30 % across the class. So every quantity below is first
computed **per event** (averaged over that event's oversamples) and then averaged over events:

- a sum ($dN/dy$, a histogram bin): the mean over events, error $\sigma/\sqrt{N_\text{ev}}$;
- a ratio ($\langle p_T\rangle$): $\sum_k S_k / \sum_k N_k$, jackknife error over events.

These errors include the oversampling noise. With a single event there is no error.

In [ ]:
def ev_hist(h, x, edges, mask=None, w=None):
    """Per-event, per-oversample histogram of x: -> (n_events, n_bins)."""
    m = np.ones(len(h.pid), bool) if mask is None else mask
    ev_edges = np.arange(h.n_events + 1) - 0.5
    H, _ = np.histogramdd((h.event[m], x[m]), bins=(ev_edges, edges),
                          weights=None if w is None else w[m])
    return H[h.good] / h.n_os

def ev_total(h, mask=None, w=None):
    """Per-event, per-oversample sum (default: count): -> (n_events,)."""
    m = np.ones(len(h.pid), bool) if mask is None else mask
    t = np.bincount(h.event[m], weights=None if w is None else w[m], minlength=h.n_events)
    return t[h.good] / h.n_os

def mean_err(rows):
    """Mean over events of a per-event sum, and its standard error."""
    rows = np.asarray(rows, float)
    n = len(rows)
    return rows.mean(0), (rows.std(0, ddof=1) / np.sqrt(n) if n > 1 else np.full(rows.shape[1:], np.nan))

def ratio_err(num, den):
    """sum_k num_k / sum_k den_k, with its jackknife error over events."""
    num, den = np.asarray(num, float), np.asarray(den, float)
    n = len(num)
    with np.errstate(invalid="ignore", divide="ignore"):
        r = num.sum(0) / den.sum(0)
        if n < 2:
            return r, np.full_like(r, np.nan)
        loo = np.array([(num.sum(0) - num[k]) / (den.sum(0) - den[k]) for k in range(n)])
    return r, np.sqrt((n - 1) / n * ((loo - loo.mean(0)) ** 2).sum(0))

def mean_pt(h, mask):
    return ratio_err(ev_total(h, mask, h.pt), ev_total(h, mask))

## 1. The data and the event classes

**How 0–10 % is selected.** In both Glauber models it is $b \in [0, 4.7]$ fm, sampled with
$P(b) \propto b$. 4.7 fm is the geometric 0–10 %: $\pi b^2 = 0.1\,\sigma_{AuAu}$, with
$\sigma_{AuAu} \approx 690$ fm². The 3D MC-Glauber has a centrality cut on its string count
(`cenMin`/`cenMax`), but **X-SCAPE never applies it**. `MCGlauberWrapper` calls
`generate_pre_events()`, which only samples $b$ on `[b_min, b_max]`. So the MUSIC XML now sets
`b_max` = 4.7; with its old `b_max` = 20 it produced minimum-bias events. Selecting on $b$ in
both models keeps them comparable. It is not how an experiment selects centrality (on
multiplicity), which gives a somewhat more central, less fluctuating class. The two models'
$\sigma_{NN}$ differ slightly: 40.8 mb in 3dMCGlauber, 42 mb in fast_data.

FastHydro events whose freeze-out surface did not close are dropped here, and nowhere else.

In [ ]:
H, META = {}, {}
for m, (d, stem) in FILES.items():
    npz, js = os.path.join(d, f"hadrons_{stem}.npz"), os.path.join(d, f"{stem}_events.json")
    if not (os.path.exists(npz) and os.path.exists(js)):
        continue
    META[m] = json.load(open(js))
    h = load(npz, META[m]["n_oversample"])
    ev = META[m].get("events") or []
    if m == "fasthydro_b0":
        for e in ev:            # runs older than the b/npart record: b is the YAML's fixed 0
            e.setdefault("b", 0.0)
    closed = [e.get("closure", {}).get("closed", True) if e.get("closure") else True for e in ev]
    closed += [True] * (h.n_events - len(closed))          # MUSIC closes its own surface
    h.good = np.array(closed[:h.n_events], bool)
    h.name = m
    H[m] = h

if not H:
    print(f"No model files under {os.path.abspath(OUT)}.\n")
    print("Generate them from your X-SCAPE build tree:\n")
    print("    python ../external_packages/js-contrib/contribs/FastHydro/example/"
          "make_bulk_comparison_data.py --device cuda\n")
    print("    # add  --music-variants calibrated,matched  to separate IC from transport\n")
    print("then run this notebook from that build tree, or point it at the output:\n")
    print("    BULK_VS_MUSIC_OUT=<build>/out_bulk_vs_music jupyter lab bulk_vs_music.ipynb")
    raise SystemExit("generate the files above, then re-run this cell")
if REF not in H:
    REF = next(iter(H))
    print(f"(no MUSIC calibrated run: ratios are to {MODELS[REF]['label']})")

print(f"{'model':42s} {'events':>7s} {'used':>5s} {'oversamples':>12s}   event class")
for m, h in H.items():
    ev = [e for e, g in zip(META[m].get("events") or [], h.good) if g]
    if "b" in (ev[0] if ev else {}):
        b = np.array([e["b"] for e in ev])
        npart = np.array([e.get("npart", np.nan) for e in ev])
        cls = f"b = {b.min():.2f}-{b.max():.2f} fm (mean {b.mean():.2f})"
        cls += f", <Npart> = {np.nanmean(npart):.0f}" if np.isfinite(npart).any() else ""
    elif "n_strings" in (ev[0] if ev else {}):
        ns = np.array([e["n_strings"] for e in ev])
        cls = f"N_strings = {ns.min()}-{ns.max()} (mean {ns.mean():.0f})"
    else:
        cls = ""
    print(f"{MODELS[m]['label']:42s} {h.n_events:7d} {h.good.sum():5d} {h.n_os:12d}   {cls}")

In [ ]:
# per-event multiplicity: the spread inside the class, and what drives it in each model
Y = 0.5
nch_ev = {m: ev_total(h, h.charged & (np.abs(h.eta) < Y)) / (2 * Y) for m, h in H.items()}

fig, axs = plt.subplots(1, 2, figsize=(12.5, 4.0))
ax = axs[0]
for i, (m, v) in enumerate(nch_ev.items()):
    x = i + np.linspace(-0.15, 0.15, len(v)) if len(v) > 1 else np.array([i])
    s = MODELS[m]
    ax.plot(x, v, s["marker"], color=s["color"], ms=5, alpha=0.75, ls="none")
    mu, e = mean_err(v)
    ax.errorbar(i + 0.3, mu, e, color=s["color"], marker="_", ms=16, mew=2, capsize=0, lw=2)
ax.set_xticks(range(len(nch_ev)), [MODELS[m]["label"].replace(", ", ",\n") for m in nch_ev],
              fontsize=8)
tidy(ax, None, r"$dN_{ch}/d\eta$,  $|\eta|<0.5$", "Per event (points) and the class mean")

ax = axs[1]
for m in ("fasthydro", "fasthydro_b0"):
    if m in H:
        ev = [e for e, g in zip(META[m]["events"], H[m].good) if g]
        ax.plot([e["b"] for e in ev], nch_ev[m], MODELS[m]["marker"], color=MODELS[m]["color"],
                ms=6, ls="none", label=MODELS[m]["label"])
for m in ("music_calibrated", "music_matched"):
    if m in H:
        mu, e = mean_err(nch_ev[m])
        ax.axhspan(mu - e, mu + e, color=MODELS[m]["color"], alpha=0.15, lw=0)
        ax.axhline(mu, **{**style(m), "lw": 1.2})
tidy(ax, r"$b$ [fm]", r"$dN_{ch}/d\eta$,  $|\eta|<0.5$",
     "FastHydro vs b; MUSIC class mean as a band")
ax.legend(frameon=False, fontsize=8)
fig.tight_layout()

## 2. Mid-rapidity yields and $\langle p_T \rangle$

The data column is a yardstick, not a target. Neither model has a hadronic cascade.
FastHydro and MUSIC matched have no $\delta f$. And none of them evolves net baryon density
(`Include_Rhob` = 0), so $\bar p/p = 1$ where data has ~0.73.

**The data values are approximate.** They are averages of the published 0–5 % and 5–10 %
PHENIX values, and $\langle p_T\rangle$ is 0–5 %; the sources are in `DATA`. Check them against
the tables before quoting anything.

In [ ]:
# 0-10% Au+Au 200 GeV, mid-rapidity.  APPROXIMATE -- check against the tables before quoting.
#   dN_ch/deta: PHENIX, PRC 71, 034908 (2005): 687 (0-5%), 560 (5-10%)  -> mean of the two
#   dN/dy, <pT>: PHENIX, PRC 69, 034909 (2004); protons corrected for weak-decay feed-down,
#                which matches iSS here (hyperons are not decayed).  dN/dy: mean of 0-5% and
#                5-10%; <pT>: 0-5%.
DATA = {
    "dNch/deta": 0.5 * (687 + 560),
    "dN/dy": {"pi+": 0.5 * (286.4 + 239.6), "pi-": 0.5 * (281.8 + 238.9),
              "K+": 0.5 * (48.9 + 40.1), "K-": 0.5 * (45.7 + 37.8),
              "p": 0.5 * (18.4 + 15.3), "pbar": 0.5 * (13.5 + 11.1)},
    "<pT>": {"pi+": 0.451, "pi-": 0.455, "K+": 0.670, "K-": 0.677, "p": 0.949, "pbar": 0.959},
}
NAMES = ("pi+", "pi-", "K+", "K-", "p", "pbar")

rows = {}          # (quantity) -> {model: (value, err)}
rows["dN_ch/deta |eta|<0.5"] = {m: mean_err(v) for m, v in nch_ev.items()}
for n in NAMES:
    rows[f"dN/dy {n}"] = {m: mean_err(ev_total(h, h.species(n) & (np.abs(h.y) < Y)) / (2 * Y))
                          for m, h in H.items()}
for n in NAMES:
    rows[f"<pT> {n} [GeV]"] = {m: mean_pt(h, h.species(n) & (np.abs(h.y) < Y)) for m, h in H.items()}
data_col = {"dN_ch/deta |eta|<0.5": DATA["dNch/deta"],
            **{f"dN/dy {n}": DATA["dN/dy"][n] for n in NAMES},
            **{f"<pT> {n} [GeV]": DATA["<pT>"][n] for n in NAMES}}

short = {"music_calibrated": "MUSIC cal", "music_matched": "MUSIC match",
         "fasthydro": "FH 0-10%", "fasthydro_b0": "FH b=0"}
ms = list(H)
print(f"{'':24s}" + "".join(f"{short[m]:>18s}" for m in ms) + f"{'PHENIX~':>10s}"
      + "".join(f"{short[m] + '/ref':>17s}" for m in ms if m != REF))
for q, r in rows.items():
    line = f"{q:24s}"
    for m in ms:
        v, e = r[m]
        prec = 3 if "<pT>" in q else 1
        line += f"{v:11.{prec}f} ±{e:5.{prec}f}" if np.isfinite(e) else f"{v:18.{prec}f}"
    line += f"{data_col[q]:10.3g}"
    for m in ms:
        if m != REF:
            line += f"{r[m][0] / r[REF][0]:17.3f}"
    print(line)
print(f"\nref = {MODELS[REF]['label']}")

## 3. $dN_{ch}/d\eta$

The longitudinal profile is the part of the realistic IC that was never looked at: its shape
comes from the Bozek–Wyskiel envelope (`eta0` = 1.5, `sig_eta` = 1.3). The 3D MC-Glauber gets
its shape from string rapidity loss. The right panel removes the normalization, so it shows
the shape alone.

FastHydro's grid ends at $|\eta_s| = 5$ with open edges, so its hadrons beyond $|\eta| \approx 4$
are cut. That region is shaded.

In [ ]:
eta_edges = np.linspace(-5.0, 5.0, 41)
etc, deta = centers(eta_edges), np.diff(eta_edges)
dndeta = {m: mean_err(ev_hist(h, h.eta, eta_edges, h.charged) / deta) for m, h in H.items()}

fig, axs = plt.subplots(1, 3, figsize=(15, 4.2))
for m, (v, e) in dndeta.items():
    band(axs[0], etc, v, e, **style(m))
    r, re = v / dndeta[REF][0], e / dndeta[REF][0]
    if m != REF:
        band(axs[1], etc, r, re, **style(m))
    i0 = np.argmin(np.abs(etc)) - 1                     # the two bins around eta = 0
    n0 = v[i0:i0 + 2].mean()
    band(axs[2], etc, v / n0, e / n0, **style(m))
axs[0].errorbar([0], [DATA["dNch/deta"]], [0.05 * DATA["dNch/deta"]], color="0.35", marker="*",
                ms=9, ls="none", capsize=0, label="PHENIX 0-10% (approx.)")
axs[1].axhline(1, color="0.5", lw=0.8)
tidy(axs[0], r"$\eta$", r"$dN_{ch}/d\eta$", r"Charged-hadron $dN/d\eta$")
tidy(axs[1], r"$\eta$", f"ratio to {MODELS[REF]['label'].split(',')[0]}", "Ratio to the reference")
tidy(axs[2], r"$\eta$", r"$dN_{ch}/d\eta$ / its value at $\eta = 0$", "Shape only")
for ax in axs:
    ax.set_xlim(eta_edges[0], eta_edges[-1])
    for s in (-1, 1):
        ax.axvspan(4 * s, 5 * s, color="0.5", alpha=0.10, lw=0)
axs[0].legend(frameon=False, fontsize=8, loc="lower center")
fig.tight_layout()

def width(v):
    """Full width at half maximum of dN/deta, by linear interpolation from the centre out."""
    half = 0.5 * v[np.abs(etc) < 0.5].mean()
    out = []
    for side in (etc > 0, etc < 0):
        x, y = np.abs(etc[side]), v[side]
        o = np.argsort(x); x, y = x[o], y[o]
        k = np.nonzero(y < half)[0]
        out.append(np.interp(half, y[k[0]:k[0] - 2:-1], x[k[0]:k[0] - 2:-1]) if len(k) and k[0] > 0 else np.nan)
    return np.nanmean(out)

print(f"{'':42s} {'dN/deta(0)':>11s} {'FWHM':>7s} {'(|eta|~3)/(0)':>14s}")
for m, (v, e) in dndeta.items():
    n0 = v[np.abs(etc) < 0.5].mean()
    n3 = v[(np.abs(etc) > 2.75) & (np.abs(etc) < 3.25)].mean()
    print(f"{MODELS[m]['label']:42s} {n0:11.1f} {width(v):7.2f} {n3 / n0:14.3f}")

## 4. $p_T$ spectra

Identified hadrons at $|y| < 0.5$, and charged hadrons at $|\eta| < 0.5$. The ratio panels show
how the spectra differ in slope as well as in normalization. A ratio that tilts with $p_T$
means different radial flow; a flat offset is multiplicity only.

In [ ]:
pt_edges = np.r_[np.arange(0.0, 2.0, 0.1), np.arange(2.0, 3.01, 0.25)]   # coarser where it is sparse
ptc, dpt = centers(pt_edges), np.diff(pt_edges)
panels = [("pi", r"$\pi^\pm$", "y"), ("K", r"$K^\pm$", "y"), ("p+pbar", r"$p+\bar p$", "y"),
          ("charged", r"charged hadrons", "eta")]
fig, axs = plt.subplots(2, 4, figsize=(16, 6.4), sharex=True,
                        gridspec_kw=dict(height_ratios=(2.2, 1)))
SPEC = {}
for j, (name, lab, rap) in enumerate(panels):
    norm = 2 * np.pi * ptc * dpt * (2 * Y)
    for m, h in H.items():
        sel = h.species(name) & (np.abs(getattr(h, rap)) < Y)
        SPEC[m, name] = mean_err(ev_hist(h, h.pt, pt_edges, sel) / norm)
    for m in H:
        v, e = SPEC[m, name]
        band(axs[0, j], ptc, v, e, **style(m))
        if m != REF:
            vr = np.where(SPEC[REF, name][0] > 0, SPEC[REF, name][0], np.nan)   # no ratio on an empty bin
            band(axs[1, j], ptc, v / vr, e / vr, **style(m))
    axs[0, j].set_yscale("log")
    axs[1, j].axhline(1, color="0.5", lw=0.8)
    axs[1, j].set_ylim(0.4, 1.6)
    tidy(axs[0, j], None, r"$\frac{1}{2\pi p_T}\frac{dN}{dp_T\,dy}$ [GeV$^{-2}$]" if j == 0 else None,
         lab + (f",  $|y|<{Y}$" if rap == "y" else f",  $|\\eta|<{Y}$"))
    tidy(axs[1, j], r"$p_T$ [GeV]", "ratio to ref" if j == 0 else None)
axs[0, 0].legend(frameon=False, fontsize=8)
fig.suptitle(f"Spectra; ratios to {MODELS[REF]['label']}", x=0.01, ha="left", fontsize=12)
fig.tight_layout()

## 5. $\langle p_T \rangle$

$\langle p_T\rangle$ rising with mass is radial flow. Two things set it: how much the fireball
expands, and how compact and lumpy its initial state is. For the initial state that means the
transverse hot-spot width `w` = 0.4 fm and the start time $\tau_0$: 0.58 fm/c with zero flow in
FastHydro, against strings that deposit from 0.42 fm/c onward in MUSIC. Compare MUSIC matched
with FastHydro to see the initial state's part; MUSIC calibrated adds bulk viscosity, which
lowers $\langle p_T\rangle$, and shear $\delta f$.

In [ ]:
MASS = {"pi": 0.1396, "K": 0.4937, "p+pbar": 0.9383}
fig, axs = plt.subplots(1, 2, figsize=(12.5, 4.2))
ax = axs[0]
for k, m in enumerate(H):
    h = H[m]
    vals = [mean_pt(h, h.species(n) & (np.abs(h.y) < Y)) for n in MASS]
    x = np.array(list(MASS.values())) + 0.015 * (k - (len(H) - 1) / 2)     # side by side
    ax.errorbar(x, [v for v, _ in vals], [e for _, e in vals], **style(m, marker=True), capsize=0)
dat = [0.5 * (DATA["<pT>"][a] + DATA["<pT>"][b]) for a, b in (("pi+", "pi-"), ("K+", "K-"), ("p", "pbar"))]
ax.plot(list(MASS.values()), dat, "*", color="0.35", ms=10, ls="none", label="PHENIX 0-5% (approx.)")
ax.set_xticks(list(MASS.values()), [r"$\pi$", "K", "p"])
tidy(ax, "hadron mass", r"$\langle p_T\rangle$ [GeV],  $|y|<0.5$", r"$\langle p_T\rangle$ vs mass")
ax.legend(frameon=False, fontsize=8)

ax = axs[1]
for m, h in H.items():
    s = ev_hist(h, h.eta, eta_edges, h.charged, h.pt)
    n = ev_hist(h, h.eta, eta_edges, h.charged)
    v, e = ratio_err(s, n)
    band(ax, etc, v, e, **style(m))
for sgn in (-1, 1):
    ax.axvspan(4 * sgn, 5 * sgn, color="0.5", alpha=0.10, lw=0)
ax.set_xlim(eta_edges[0], eta_edges[-1])
tidy(ax, r"$\eta$", r"$\langle p_T\rangle_{ch}$ [GeV]", r"Charged-hadron $\langle p_T\rangle$ vs $\eta$")
fig.tight_layout()

## 6. What this says about the IC tuning

The numbers below turn the comparison into first-order changes to
`fasthydro_wake_realistic.yaml`. They are compared with the reference model, and with data for
the multiplicity.

- **Normalization.** Multiplicity is proportional to the total entropy, which scales with the
  normalization $K$ (`quantity: entropy`). Near $T \approx 0.39$ GeV, $s \propto T^3$ is a fair
  approximation, so $T_\text{target} \propto K^{1/3}$. Two caveats: viscous entropy production
  changes with the temperature, and the calibration is at $b = 0$ while the comparison is
  0–10 %. Treat the result as a starting point for one more iteration, not the answer.
- **Longitudinal shape.** A narrower $dN/d\eta$ than the reference means the envelope is too
  short: raise `eta0` (the plateau half-width) or `sig_eta` (the Gaussian fall-off). A wider
  one means lower them.
- **$\langle p_T\rangle$.** If FastHydro is harder than MUSIC *matched*, the fast_data IC is too
  compact or too lumpy. The knobs are the hot-spot width `w` and $\tau_0$.

In [ ]:
def ratio(q, m, ref=REF):
    return rows[q][m][0] / rows[q][ref][0]

T0 = 0.39
fh = [m for m in ("fasthydro", "fasthydro_b0") if m in H]
if not fh:
    print("no FastHydro run loaded")
for m in fh:
    print(f"== {MODELS[m]['label']} ==")
    targets = [(MODELS[REF]["label"], rows["dN_ch/deta |eta|<0.5"][REF][0])] if REF != m else []
    targets.append(("PHENIX 0-10% (approx.)", DATA["dNch/deta"]))
    n_fh = rows["dN_ch/deta |eta|<0.5"][m][0]
    for lab, n in targets:
        k = n / n_fh
        print(f"  dN_ch/deta {n_fh:6.1f} vs {n:6.1f} ({lab}):  K x {k:.3f}  ->  "
              f"target_T ~ {T0 * k ** (1 / 3):.3f} GeV  (now {T0})")
    if REF != m:
        v, _ = dndeta[m]; vr, _ = dndeta[REF]
        print(f"  dN/deta FWHM {width(v):.2f} vs {width(vr):.2f}   "
              f"(|eta|~3)/(0): {v[(np.abs(etc) > 2.75) & (np.abs(etc) < 3.25)].mean() / v[np.abs(etc) < 0.5].mean():.3f}"
              f" vs {vr[(np.abs(etc) > 2.75) & (np.abs(etc) < 3.25)].mean() / vr[np.abs(etc) < 0.5].mean():.3f}")
        for n in ("pi+", "K+", "p"):
            line = f"  <pT> {n:4s} / ref = {ratio(f'<pT> {n} [GeV]', m):.3f}"
            if "music_matched" in H and "music_matched" != REF:
                line += f"   / MUSIC matched = {ratio(f'<pT> {n} [GeV]', m, 'music_matched'):.3f}"
            print(line)

### Reading the result (the run made alongside this notebook: 25 events × 100 oversamples per model)

At mid-rapidity, 0–10 %, with errors from the event average:

| | $dN_{ch}/d\eta$ | $\langle p_T\rangle$ $\pi$ / K / p [GeV] | $dN/d\eta$ FWHM |
|---|---|---|---|
| MUSIC calibrated | 667 ± 18 | 0.462 / 0.666 / 0.962 | 3.51 |
| MUSIC matched | 544 ± 15 | 0.526 / 0.762 / 1.039 | 3.53 |
| FastHydro 0–10 % | 497 ± 12 | 0.508 / 0.731 / 1.002 | 3.56 |
| FastHydro $b = 0$ | 634 ± 3 | 0.513 / 0.737 / 1.011 | 3.56 |
| PHENIX (approx.) | ~624 | 0.45 / 0.67 / 0.95 | |

- **The initial state alone (FastHydro vs MUSIC matched) is close.**
  - Multiplicity is 9 % low (497 against 544).
  - $\langle p_T\rangle$ is 3–4 % softer for every species. Relative to MUSIC matched, the
    spectrum ratio falls with $p_T$, from ~0.97 at low $p_T$ to ~0.8 at 2.5 GeV. So the fast_data
    IC gives somewhat less radial flow; it is not too compact.
  - The $dN/d\eta$ shape agrees to 1 % in FWHM. FastHydro's tails are ~5 % higher at
    $|\eta| \approx 3$, so `eta0` / `sig_eta` need at most a small trim.
- **The normalization is where the realistic config is off.** `target_T` = 0.39 was set at
  $b = 0$. Over 0–10 % ($\langle b\rangle$ = 3.4 fm) it gives 497, 20 % below the data's ~624
  and 9 % below the 3D Glauber's matched value. To first order, 0–10 % needs `target_T` ≈ 0.42
  for the data or ≈ 0.40 for MUSIC matched. The $b = 0$ events, where the wake runs live, give
  634, 8 % under PHENIX's 0–5 % (687).
- **The calibrated transport is a larger effect than the initial state.** Bulk viscosity and
  $\delta f$ raise the multiplicity by 23 % (entropy production) and lower
  $\langle p_T\rangle$ by 7–13 %. That brings MUSIC's $\langle p_T\rangle$ to within
  ~2 % of data. FastHydro's 5–13 % excess over data is therefore transport, not IC.
- **Protons.** MUSIC calibrated has 25.6 $p$ per unit rapidity against ~17 in data, and
  $\bar p = p$. There is no SMASH (no $p\bar p$ annihilation) and no net baryon density, so
  proton yields are not a test of either IC here.

## 7. What this does not include

- **Centrality on $b$, not on multiplicity.** Both Glauber models cut on $b$, which keeps them
  comparable (§1). Against data, a multiplicity-selected 0–10 % is somewhat more central.
- **No hadronic cascade.** SMASH is not built, so none of the three runs has hadronic
  rescattering. The calibrated 3D-Glauber parameters were tuned *with* SMASH, so even MUSIC
  calibrated is not the calibrated model's final answer. Rescattering hardens protons and
  softens pions.
- **EOS 9, not 91.** iSS decays resonances only for the UrQMD list, so both hydros run EOS 9.
  It differs from EOS 91 only below $T_c$.
- **No net baryon density.** The 3D MC-Glauber transports baryon number, but `Include_Rhob` = 0
  throws it away, so $\mu_B = 0$ at freeze-out in all runs.
- **Different pre-hydro dynamics.** The 3D Glauber deposits its strings as dynamical sources
  from $\tau_\text{form} = 0.42$ fm/c on. The fast_data IC is a static profile at 0.58 fm/c with
  zero flow. This is part of "the initial state" as compared here.
- **FastHydro's η edges.** Its grid ends at $|\eta_s| = 5$, so beyond $|\eta| \approx 4$ its
  $dN/d\eta$ is cut short.